# Validation statistics and Visualizations for CRS & training set 

## Inspecting the CRS data

In [ ]:
# Load from feather
import pandas as pd

crs_raw = pd.read_feather("../../data/raw/crs_raw.feather")

In [ ]:
# Alphabetically sorted columns
sorted(crs_raw.columns)

crs_raw.shape

In [ ]:
# Histogram of year column with bin size = 1 year
print(crs_raw['year'].min(), crs_raw['year'].max())
crs_raw['year'].hist(bins=range(crs_raw['year'].min(), crs_raw['year'].max() + 2), edgecolor='black')

import matplotlib.pyplot as plt
plt.title("Number of projects by year")
plt.xlabel("Year")
plt.ylabel("Frequency")
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Set ticks to start at the first year and end at the last year
plt.xticks(range(crs_raw['year'].min(), crs_raw['year'].max() + 1, 5))  # Adjust step size (e.g., 5) as needed
plt.show()


#### Evaluation of purpose code use over time 

In [ ]:
# Count of values in the purpose_code column for each year
counts_16062 = crs_raw[crs_raw['purpose_code'] == 16062].groupby('year')['purpose_code'].count()
counts_15250 = crs_raw[crs_raw['purpose_code'] == 15250].groupby('year')['purpose_code'].count()
counts_15170 = crs_raw[crs_raw['purpose_code'] == 15170].groupby('year')['purpose_code'].count()
counts_15180 = crs_raw[crs_raw['purpose_code'] == 15180].groupby('year')['purpose_code'].count()

# Plot the counts for all purpose codes
plt.figure(figsize=(12, 6))
plt.plot(counts_16062.index, counts_16062.values, label=f'16062 - Stat. cap (total {counts_16062.sum()})', marker='o')
plt.plot(counts_15250.index, counts_15250.values, label=f'15250 - mining (total {counts_15250.sum()})', marker='o')
plt.plot(counts_15170.index, counts_15170.values, label=f'15170 - womens equality (total {counts_15170.sum()})', marker='o')
plt.plot(counts_15180.index, counts_15180.values, label=f'15180 - ending violence against women (total {counts_15180.sum()})', marker='o')

# Add title, labels, and legend
plt.title('Counts of Purpose Codes by Year')
plt.xlabel('Year')
plt.ylabel('Count')
plt.legend()
plt.show()

#### Long description statistics

In [ ]:
# Summary statistics for long description with number of na values
print(crs_raw["long_description"].isna().sum())
print(crs_raw["long_description"].describe())

# Mean length of long descriptions
crs_raw["long_description"].str.len().mean()

In [ ]:
import matplotlib.pyplot as plt 
# Histogram of long description lengths
pd.Series(crs_raw["long_description"].unique()).str.len().hist(bins=100)

plt.title(f"Histogram of Long Description Lengths (total unique long descriptions: {crs['long_description'].nunique()})")
plt.xlabel("Number of characters in long description")
plt.ylabel("Frequency")
plt.show()

In [ ]:
import os 
import pandas as pd

# Save first 100 non-na long descriptions to csv in folder data/development/
if not os.path.exists("../../data/development/"):
    os.makedirs("../../data/development/")

# Convert the unique values to a pandas DataFrame and save to CSV
pd.DataFrame(crs_raw["long_description"].dropna().unique()[:100], columns=["long_description"]).to_csv(
    "../../data/development/crs_test_long_description.csv", index=False
)

#### Short description statistics

In [ ]:
# Make a histogram of the unique short_description lengths and include the number of None values
pd.Series(crs_raw["short_description"].unique()).str.len().hist(bins=80)

import matplotlib.pyplot as plt
plt.title(f"Histogram of Short Description Lengths (total unique short descriptions: {crs_raw['short_description'].nunique()}, None values: {crs_raw['short_description'].isna().sum()})")
plt.xlabel("Number of characters in short description")
plt.ylabel("Frequency")
plt.show()

#### Project title statistics

In [ ]:
# Make a histogram of the unique prject_title lengths and include the number of None values
pd.Series(crs["project_title"].unique()).str.len().hist(bins=100)

import matplotlib.pyplot as plt
plt.title(f"Histogram of Project Title Lengths (total unique project titles: {crs['project_title'].nunique()}, None values: {crs['project_title'].isna().sum()})")
plt.xlabel("Number of characters in project title")
plt.ylabel("Frequency")
plt.show()

## Training set inspection (after A - Title Pattern Matching)

In [ ]:
# Load data after title matching
crs_test = pd.read_feather("../../data/processed/crs_title_stat_matched.feather")

In [ ]:
# Manual inspection in xlsx
# Sample randomly 25% of the rows
crs_sampled = crs_test.sample(frac=0.25, random_state=42)

with pd.ExcelWriter("../../data/processed/crs_title_stat_matched.xlsx", engine="openpyxl") as writer:
    crs_sampled.to_excel(writer, index=False)

del crs_sampled

#### Language distribution

In [ ]:
# For every language, how many projects have a title_stat_match?
language_distr = crs_test.groupby('language')['title_stat_match'].sum()

# Remove languages with value < 2
language_distr = language_distr[language_distr >= 2]

# Sort the language distribution in descending order
language_distr = language_distr.sort_values(ascending=False)

# Make a bar plot of the language distribution
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
bars = plt.bar(language_distr.index, language_distr.values, color='skyblue')
plt.title('Number of Projects with Title Stat Match by Language')
plt.xlabel('Language')
plt.ylabel('Number of Projects')
plt.tight_layout()

# Add values on top of the bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height, str(height), ha='center', va='bottom', fontsize=9)

plt.show()

#### Keyword analysis

In [ ]:
# Filter for English projects
matched_keywords_frquency = crs_test['matched_keywords'].explode().value_counts()

import matplotlib.pyplot as plt

# Plot the bar chart
ax = matched_keywords_frquency.plot(kind='barh', figsize=(10, 20))
plt.title(f"Matched Keywords project titles in English (total matched {matched_keywords_frquency.sum()})")
plt.xlabel("Frequency")
plt.ylabel("Matched Keywords")

# Add frequency values to the right of each bar
for i, v in enumerate(matched_keywords_frquency):
    ax.text(v + 1, i, str(v), va='center', fontsize=9)

plt.show()